# Prompt A/B Test: four wordings of one request, compared side by side

Send the same paragraph to the model with four different summarization prompts, trace every call in LangSmith, and compare the answers in one table. Only the instruction changes between calls, so any difference in the output comes from the wording.

Run the cells top to bottom (**Shift+Enter** in Jupyter or VS Code).

## Step 1: Install the packages and import the libraries

In [ ]:
%pip install -q langchain langchain-openai langchain-community langchain-google-genai tabulate python-dotenv

In [2]:
import os
from getpass import getpass
from pathlib import Path

from dotenv import dotenv_values
from tabulate import tabulate
from langchain_openai import AzureChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

## Step 2: Configure the model

The original lab targets a course-provided Azure OpenAI endpoint (`gpt-5-mini`). This cell tries Azure first if you supply a key, and otherwise uses **Google Gemini**.

Keys are read from a local `.env` file (see `.env.example`) and are never printed.

In [4]:
# OPTIONAL: paste an AZURE_OPENAI_API_KEY between the quotes, or set it in .env.
# Leave it empty to use Gemini.
AZURE_KEY = ""

# ---------- Load keys from a local .env file (nothing secret is printed) ----------
saved = dotenv_values(Path(".env"))

# ---------- LangSmith tracing, in its own project so it stays separate from other builds ----------
if saved.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = saved["LANGSMITH_API_KEY"]
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "Demo-02-AB-Prompt-Test"
else:
    os.environ["LANGSMITH_TRACING"] = "false"

# ---------- Azure Chat Model (the course's model) ----------
# The endpoint URL is written as three separate strings so that no part of it
# can be broken by line wrapping. Python joins them into a single URL.
AZURE_ENDPOINT = (
    "https://openai-api-management-gw.azure-api.net"
    "/openaiprodtest-clone/deployments/gpt-5-mini/chat/completions"
    "?api-version=2024-12-01-preview"
)

llm = None
model_label = None

azure_key = AZURE_KEY.strip() or saved.get("AZURE_OPENAI_API_KEY") or ""
if azure_key:
    os.environ["AZURE_OPENAI_API_KEY"] = azure_key
    os.environ.setdefault("OPENAI_API_TYPE", "azure")
    os.environ.setdefault("OPENAI_API_VERSION", "2025-01-01-preview")
    try:
        azure_llm = AzureChatOpenAI(
            azure_endpoint=AZURE_ENDPOINT,
            api_version="2024-12-01-preview",
            deployment_name="gpt-5-mini",
            temperature=1,
            timeout=30,
            max_retries=1,
        )
        azure_llm.invoke([HumanMessage(content="Reply with the single word OK.")])
        llm, model_label = azure_llm, "Azure OpenAI gpt-5-mini"
    except Exception as err:
        print("Azure is not reachable from this PC:", type(err).__name__, "-", str(err)[:120])
        print("Switching to Gemini instead.")

# ---------- Google Gemini (fallback) ----------
if llm is None:
    google_key = saved.get("GOOGLE_API_KEY") or os.getenv("GOOGLE_API_KEY") or ""
    if not google_key:
        google_key = getpass("Paste your GOOGLE_API_KEY and press Enter: ")
    os.environ["GOOGLE_API_KEY"] = google_key.strip()

    gemini_model = saved.get("MODEL_NAME") or "gemini-3.5-flash-lite"
    llm = ChatGoogleGenerativeAI(
        model=gemini_model,
        temperature=1,
        max_retries=6,  # free-tier Gemini keys are rate limited; retry politely
    )
    model_label = f"Google {gemini_model}"

print("Model in use:", model_label)
print("LangSmith tracing:", os.environ["LANGSMITH_TRACING"],
      "| Project:", os.getenv("LANGSMITH_PROJECT", "-"))

Model in use: Google gemini-3.5-flash-lite
LangSmith tracing: true | Project: Demo-02-AB-Prompt-Test


## Step 3: Define the prompt variants and the input text

`text_to_summarize` is the paragraph. `prompt_variants` holds four ways of asking for a summary. Everything else stays the same, so any difference in the answers comes from the wording of the prompt.

In [5]:
# ---------- Text to Summarize ----------
text_to_summarize = (
    "Artificial Intelligence is revolutionizing industries by enabling machines "
    "to learn from data, adapt to new inputs, and perform tasks that normally "
    "require human intelligence."
)

# ---------- Prompt Variants ----------
prompt_variants = [
    "Summarize the following in one sentence:\n",
    "Write a short summary of this paragraph:\n",
    "Condense this into a concise statement:\n",
    "TL;DR:\n"
]

## Step 4: Compare the outputs across prompt variants

This sends four requests to the model, one per variant, and prints a comparison table.

In [6]:
def text_of(message) -> str:
    """Return the reply as plain text (Gemini can return a list of parts)."""
    content = message.content
    if isinstance(content, str):
        return content
    return "".join(
        part.get("text", "") if isinstance(part, dict) else str(part)
        for part in content
    )


# ---------- Run Variants ----------
results = []

for idx, prefix in enumerate(prompt_variants):
    full_prompt = prefix + text_to_summarize

    response = llm.invoke(
        [HumanMessage(content=full_prompt)],
        config={"run_name": f"Variant {idx+1}", "tags": ["demo-02", "ab-test"]},
    )
    answer = text_of(response).strip()

    results.append([
        f"Variant {idx+1}",
        prefix.strip(),
        answer,
        len(answer.split()),
    ])

# ---------- Display Results ----------
print("Model:", model_label, "\n")
print(tabulate(
    results,
    headers=["Variant", "Prompt", "AI Response", "Words"],
    tablefmt="grid",
    maxcolwidths=[10, 26, 70, 6],
))

Model: Google gemini-3.5-flash-lite 

+-----------+----------------------------+-----------------------------------------------------------------------+---------+
| Variant   | Prompt                     | AI Response                                                           |   Words |
+===========+============================+=======================================================================+=========+
| Variant 1 | Summarize the following in | Artificial Intelligence is transforming industries by empowering      |      16 |
|           | one sentence:              | machines to learn from data and perform human-like tasks.             |         |
+-----------+----------------------------+-----------------------------------------------------------------------+---------+
| Variant 2 | Write a short summary of   | **Summary:**                                                          |      17 |
|           | this paragraph:            | Artificial Intelligence is transforming indu

## What to look for

- **Length:** which wording gives the shortest answer? The `Words` column makes this easy to compare.
- **Format:** does any variant return more than one sentence, bullet points, or a label such as "Summary:"?
- **Faithfulness:** does every answer keep the three ideas from the paragraph (learn from data, adapt to new inputs, do tasks that need human intelligence)?

The saved output above is from one run on Gemini 3.5 Flash-Lite. This model uses fixed sampling settings and ignores `temperature`, and answers can still vary between runs. With only `TL;DR:` in front of the paragraph, the model read the paragraph as the user's own summary and replied conversationally instead of summarizing.